In [ ]:
import pandas as pd
from datetime import datetime

print("TrendPulse Project - Part 1")
print("Started at:", datetime.now())

TrendPulse Project - Part 1
Started at: 2026-11-03 16:10:16.882981


In [ ]:
import pandas as pd

gb = pd.read_csv("GBvideos.csv")
india = pd.read_csv("INvideos.csv")
us = pd.read_csv("USvideos.csv")

print("GB records:", len(gb))
print("IN records:", len(india))
print("US records:", len(us))

print("Total records:", len(gb) + len(india) + len(us))

GB records: 15075
IN records: 15075
US records: 16080
Total records: 46230


In [ ]:
import pandas as pd

gb = pd.read_csv("GBvideos.csv")
india = pd.read_csv("INvideos.csv")
us = pd.read_csv("USvideos.csv")
categories = pd.read_csv("categories.csv")

print("GB videos:", gb.shape)
print("India videos:", india.shape)
print("US videos:", us.shape)
print("Categories:", categories.shape)

total = len(gb) + len(india) + len(us)

print("\nTotal records:", total)

print("\nUS columns:")
print(us.columns.tolist())

GB videos: (15075, 16)
India videos: (15075, 16)
US videos: (16080, 16)
Categories: (17, 2)

Total records: 46230

US columns:
['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description']


In [ ]:
import pandas as pd

required_columns = [
    "video_id",
    "trending_date",
    "title",
    "channel_title",
    "category_id",
    "publish_time",
    "views",
    "likes",
    "dislikes",
    "comment_count"
]

def load_country(path, country, max_rows=None):

    df = pd.read_csv(path, nrows=max_rows)

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    df["country"] = country

    records = df.to_dict("records")

    print(f"{country}: {len(records)} records loaded")

    return records

In [ ]:
gb_records = load_country("GBvideos.csv", "GB")
in_records = load_country("INvideos.csv", "IN")
us_records = load_country("USvideos.csv", "US")

all_records = gb_records + in_records + us_records

print("Total records:", len(all_records))

GB: 15075 records loaded
IN: 15075 records loaded
US: 16080 records loaded
Total records: 46230


In [ ]:
print("Category columns:")
print(categories.columns.tolist())

print("\nCategory data:")
print(categories.head(10))

Category columns:
['category_id', 'category_name']

Category data:
   category_id     category_name
0            1  Film & Animation
1            2  Autos & Vehicles
2           10             Music
3           15    Pets & Animals
4           17            Sports
5           19   Travel & Events
6           20            Gaming
7           22    People & Blogs
8           23            Comedy
9           24     Entertainment


In [ ]:
categories.columns = categories.columns.str.strip()

print(categories.columns.tolist())

['category_id', 'category_name']


In [ ]:
category_names = dict(
    zip(
        categories["category_id"].astype(str),
        categories["category_name"]
    )
)

print("Category Dictionary:")
print(category_names)

Category Dictionary:
{'1': 'Film & Animation', '2': 'Autos & Vehicles', '10': 'Music', '15': 'Pets & Animals', '17': 'Sports', '19': 'Travel & Events', '20': 'Gaming', '22': 'People & Blogs', '23': 'Comedy', '24': 'Entertainment', '25': 'News & Politics', '26': 'Howto & Style', '27': 'Education', '28': 'Science & Technology', '29': 'Nonprofits & Activism', '30': 'Movies', '43': 'Shows'}


In [ ]:
import pandas as pd
from datetime import datetime

# Numeric columns
numeric_columns = [
    "views",
    "likes",
    "dislikes",
    "comment_count"
]

# Boolean columns
boolean_columns = [
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed"
]

# Count conversion failures
parse_failures = {}

for col in numeric_columns + boolean_columns + [
    "trending_date", "publish_time"
]:
    parse_failures[col] = 0


# Convert numeric values
def convert_integer(value, field):

    try:
        if pd.isna(value) or str(value).strip() == "":
            parse_failures[field] += 1
            return 0

        return int(float(str(value).replace(",", "")))

    except (ValueError, TypeError):
        parse_failures[field] += 1
        return 0


# Convert True/False values
def convert_boolean(value, field):

    try:
        if pd.isna(value):
            parse_failures[field] += 1
            return False

        value = str(value).strip().lower()

        if value in ["true", "1", "yes"]:
            return True

        if value in ["false", "0", "no"]:
            return False

        parse_failures[field] += 1
        return False

    except (ValueError, TypeError):
        parse_failures[field] += 1
        return False


# Create cleaned records
cleaned_records = []

for record in all_records:

    new_record = record.copy()

    for col in numeric_columns:
        new_record[col] = convert_integer(
            new_record[col], col
        )

    for col in boolean_columns:
        new_record[col] = convert_boolean(
            new_record[col], col
        )

    cleaned_records.append(new_record)


print("Numeric and Boolean conversion completed!")
print("Total cleaned records:", len(cleaned_records))

Numeric and Boolean conversion completed!
Total cleaned records: 46230


In [ ]:
def convert_trending_date(value):

    try:
        if pd.isna(value) or str(value).strip() == "":
            parse_failures["trending_date"] += 1
            return None

        return datetime.strptime(
            str(value).strip(), "%y.%d.%m"
        ).date()

    except (ValueError, TypeError):
        parse_failures["trending_date"] += 1
        return None


def convert_publish_time(value):

    try:
        if pd.isna(value) or str(value).strip() == "":
            parse_failures["publish_time"] += 1
            return None

        return datetime.fromisoformat(
            str(value).strip().replace("Z", "+00:00")
        )

    except (ValueError, TypeError):
        parse_failures["publish_time"] += 1
        return None


for record in cleaned_records:

    record["trending_date"] = convert_trending_date(
        record["trending_date"]
    )

    record["publish_time"] = convert_publish_time(
        record["publish_time"]
    )

print("Date conversion completed!")

Date conversion completed!


In [ ]:
print("Data type:", type(cleaned_records))

print("First record:")
print(cleaned_records[0])

print("Total records:", len(cleaned_records))

Data type: <class 'list'>
First record:
{'video_id': 'u_nKn025TZS', 'trending_date': datetime.date(2017, 11, 14), 'title': 'Wild Winter Prank', 'channel_title': 'OfficialRecipe78', 'category_id': 28, 'publish_time': datetime.datetime(2017, 10, 18, 7, 42, 2, tzinfo=datetime.timezone.utc), 'tags': '[none]', 'views': 84344, 'likes': 2155, 'dislikes': 163, 'comment_count': 475, 'thumbnail_link': 'https://i.ytimg.com/vi/u_nKn025TZS/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about budget. Business enquiries in the channel about page.', 'country': 'GB'}
Total records: 46230


In [ ]:
video_tuples = []

for record in cleaned_records:
    item = (
        record["video_id"],
        record["trending_date"],
        record["views"],
        record["likes"]
    )

    video_tuples.append(item)

print("First 5 tuples:")
print(video_tuples[:5])

print("Tuple type:", type(video_tuples[0]))

First 5 tuples:
[('u_nKn025TZS', datetime.date(2017, 11, 14), 84344, 2155), ('FoQAEGxMMQD', datetime.date(2017, 11, 14), 230434, 5412), ('wHqDBuStXq3', datetime.date(2017, 11, 14), 338852, 16813), ('ZId0IGzSaaZ', datetime.date(2017, 11, 14), 160896, 4221), ('1-xA4eRrYSH', datetime.date(2017, 11, 14), 59441, 1475)]
Tuple type: <class 'tuple'>


In [ ]:
gb_ids = {
    r["video_id"] for r in cleaned_records
    if r["country"] == "GB"
}

in_ids = {
    r["video_id"] for r in cleaned_records
    if r["country"] == "IN"
}

us_ids = {
    r["video_id"] for r in cleaned_records
    if r["country"] == "US"
}

print("GB unique videos:", len(gb_ids))
print("IN unique videos:", len(in_ids))
print("US unique videos:", len(us_ids))

GB unique videos: 4113
IN unique videos: 5878
US unique videos: 5068


In [ ]:
shared_videos = (
    (gb_ids & in_ids) |
    (gb_ids & us_ids) |
    (in_ids & us_ids)
)

unique_gb = gb_ids - (in_ids | us_ids)
unique_in = in_ids - (gb_ids | us_ids)
unique_us = us_ids - (gb_ids | in_ids)

print("Videos trending in multiple countries:", len(shared_videos))

print("GB unique videos:", len(unique_gb))
print("IN unique videos:", len(unique_in))
print("US unique videos:", len(unique_us))

Videos trending in multiple countries: 340
GB unique videos: 3873
IN unique videos: 5652
US unique videos: 4834


In [ ]:
channel_counts = {}

for record in cleaned_records:
    channel = record.get("channel_title") or "Unknown"

    if channel in channel_counts:
        channel_counts[channel] += 1
    else:
        channel_counts[channel] = 1

top_15 = sorted(
    channel_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:15]

print("Top 15 channels:")
print(top_15)

Top 15 channels:
[('QuietChallenge75', 2464), ('OfficialReview41', 1586), ('UltimateUnboxing65', 1433), ('InsaneReaction46', 1233), ('Brand NewChallenge45', 1175), ('CrazyTrailer91', 1018), ('FinalTrailer17', 911), ('UltimateSpeech20', 770), ('Brand NewSecret51', 684), ('WildHighlights7', 682), ('SurprisingHighlights41', 650), ('FinalBreakdown11', 643), ('CrazyRecipe67', 603), ('QuietInterview9', 570), ('UltimateVlog96', 565)]


In [ ]:
summary = {}

for record in cleaned_records:

    country = record["country"]

    category_id = str(int(float(record["category_id"])))

    category = category_names.get(category_id, "Unknown")

    if country not in summary:
        summary[country] = {}

    if category not in summary[country]:
        summary[country][category] = 0

    summary[country][category] += 1

print(summary)

{'GB': {'Science & Technology': 470, 'Howto & Style': 588, 'Comedy': 941, 'Entertainment': 2844, 'Music': 5013, 'Gaming': 607, 'People & Blogs': 1299, 'Autos & Vehicles': 345, 'Education': 471, 'Sports': 678, 'News & Politics': 536, 'Film & Animation': 714, 'Pets & Animals': 310, 'Travel & Events': 259}, 'IN': {'Music': 2298, 'Film & Animation': 770, 'Entertainment': 5117, 'Comedy': 966, 'People & Blogs': 1098, 'Education': 620, 'News & Politics': 1141, 'Travel & Events': 190, 'Howto & Style': 536, 'Sports': 585, 'Pets & Animals': 287, 'Autos & Vehicles': 260, 'Gaming': 660, 'Science & Technology': 547}, 'US': {'People & Blogs': 1423, 'News & Politics': 755, 'Music': 3052, 'Education': 561, 'Entertainment': 4371, 'Howto & Style': 713, 'Comedy': 1366, 'Gaming': 806, 'Travel & Events': 262, 'Sports': 667, 'Film & Animation': 824, 'Autos & Vehicles': 404, 'Pets & Animals': 321, 'Science & Technology': 555}}


In [ ]:
print("Data type:", type(cleaned_records))

print("First record:")
print(cleaned_records[0])

print("Total records:", len(cleaned_records))

Data type: <class 'list'>
First record:
{'video_id': 'u_nKn025TZS', 'trending_date': datetime.date(2017, 11, 14), 'title': 'Wild Winter Prank', 'channel_title': 'OfficialRecipe78', 'category_id': 28, 'publish_time': datetime.datetime(2017, 10, 18, 7, 42, 2, tzinfo=datetime.timezone.utc), 'tags': '[none]', 'views': 84344, 'likes': 2155, 'dislikes': 163, 'comment_count': 475, 'thumbnail_link': 'https://i.ytimg.com/vi/u_nKn025TZS/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about budget. Business enquiries in the channel about page.', 'country': 'GB'}
Total records: 46230


In [ ]:
video_tuples = []

for record in cleaned_records:

    item = (
        record["video_id"],
        record["trending_date"],
        record["views"],
        record["likes"]
    )

    video_tuples.append(item)

print("First 5 tuples:")
print(video_tuples[:5])

print("Data type:", type(video_tuples[0]))

First 5 tuples:
[('u_nKn025TZS', datetime.date(2017, 11, 14), 84344, 2155), ('FoQAEGxMMQD', datetime.date(2017, 11, 14), 230434, 5412), ('wHqDBuStXq3', datetime.date(2017, 11, 14), 338852, 16813), ('ZId0IGzSaaZ', datetime.date(2017, 11, 14), 160896, 4221), ('1-xA4eRrYSH', datetime.date(2017, 11, 14), 59441, 1475)]
Data type: <class 'tuple'>


In [ ]:
gb_ids = {
    r["video_id"] for r in cleaned_records
    if r["country"] == "GB"
}

in_ids = {
    r["video_id"] for r in cleaned_records
    if r["country"] == "IN"
}

us_ids = {
    r["video_id"] for r in cleaned_records
    if r["country"] == "US"
}

print("GB unique videos:", len(gb_ids))
print("India unique videos:", len(in_ids))
print("US unique videos:", len(us_ids))

GB unique videos: 4113
India unique videos: 5878
US unique videos: 5068


In [ ]:
# Videos trending in multiple countries

shared_videos = (
    (gb_ids & in_ids) |
    (gb_ids & us_ids) |
    (in_ids & us_ids)
)

# Videos unique to each country

unique_gb = gb_ids - (in_ids | us_ids)

unique_in = in_ids - (gb_ids | us_ids)

unique_us = us_ids - (gb_ids | in_ids)

print("Shared videos:", len(shared_videos))

print("GB unique videos:", len(unique_gb))

print("India unique videos:", len(unique_in))

print("US unique videos:", len(unique_us))

Shared videos: 340
GB unique videos: 3873
India unique videos: 5652
US unique videos: 4834


In [ ]:
channel_counts = {}

for record in cleaned_records:

    channel = record.get("channel_title") or "Unknown"

    if channel in channel_counts:
        channel_counts[channel] += 1
    else:
        channel_counts[channel] = 1

# Find top 15 channels

top_15 = sorted(
    channel_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:15]

print("Top 15 YouTube Channels:")
for channel, count in top_15:
    print(channel, ":", count)

Top 15 YouTube Channels:
QuietChallenge75 : 2464
OfficialReview41 : 1586
UltimateUnboxing65 : 1433
InsaneReaction46 : 1233
Brand NewChallenge45 : 1175
CrazyTrailer91 : 1018
FinalTrailer17 : 911
UltimateSpeech20 : 770
Brand NewSecret51 : 684
WildHighlights7 : 682
SurprisingHighlights41 : 650
FinalBreakdown11 : 643
CrazyRecipe67 : 603
QuietInterview9 : 570
UltimateVlog96 : 565


In [ ]:
summary = {}

for record in cleaned_records:

    country = record["country"]

    try:
        category_id = str(int(float(record["category_id"])))
    except (ValueError, TypeError):
        category_id = "Unknown"

    category = category_names.get(category_id, "Unknown")

    if country not in summary:
        summary[country] = {}

    if category not in summary[country]:
        summary[country][category] = 0

    summary[country][category] += 1


# Display category summary

for country, categories_data in summary.items():

    print(f"\nCountry: {country}")

    for category, count in list(categories_data.items())[:5]:
        print(f"{category}: {count}")


Country: GB
Science & Technology: 470
Howto & Style: 588
Comedy: 941
Entertainment: 2844
Music: 5013

Country: IN
Music: 2298
Film & Animation: 770
Entertainment: 5117
Comedy: 966
People & Blogs: 1098

Country: US
People & Blogs: 1423
News & Politics: 755
Music: 3052
Education: 561
Entertainment: 4371


In [ ]:
# Global variable
project_name = "TrendPulse"

def scope_demo():

    # Local variable
    message = "Welcome to Python Project"

    print("Local variable:", message)
    print("Global variable:", project_name)


# Call the function
scope_demo()

# Try accessing local variable outside
try:
    print(message)

except NameError:
    print("Local variable cannot be accessed outside the function")

Local variable: Welcome to Python Project
Global variable: TrendPulse
Local variable cannot be accessed outside the function


In [ ]:
def title_features(title):

    words = title.split()

    uppercase_words = 0
    exclamation_marks = 0
    question_marks = 0

    # Count uppercase words
    for word in words:
        if word.isupper() and any(ch.isalpha() for ch in word):
            uppercase_words += 1

    # Count ! and ? using range and index
    for i in range(len(title)):
        if title[i] == "!":
            exclamation_marks += 1

        elif title[i] == "?":
            question_marks += 1

    return {
        "uppercase_words": uppercase_words,
        "exclamation_marks": exclamation_marks,
        "question_marks": question_marks,
        "total_words": len(words)
    }


# Test the function
sample_title = "WOW! INDIA IS AMAZING??"

print(title_features(sample_title))

{'uppercase_words': 4, 'exclamation_marks': 1, 'question_marks': 2, 'total_words': 4}


In [ ]:
threshold = 1000000

i = 0
found_video = None

while i < len(cleaned_records):

    record = cleaned_records[i]
    i += 1

    # Skip videos without a title
    if not record.get("title"):
        continue

    # Skip videos below the threshold
    if record["views"] <= threshold:
        continue

    # Stop when a video above 1 million views is found
    found_video = record
    break


if found_video is not None:
    print("Video found!")
    print("Title:", found_video["title"])
    print("Views:", found_video["views"])
    print("Country:", found_video["country"])
else:
    print("No video found above 1 million views")

Video found!
Title: Lena Ortiz - Phones (Official Video)
Views: 1034753
Country: GB


In [ ]:
def classify_video(record):

    views = record.get("views", 0) or 0
    likes = record.get("likes", 0) or 0
    dislikes = record.get("dislikes", 0) or 0

    ratings_disabled = record.get("ratings_disabled", False)
    comments_disabled = record.get("comments_disabled", False)

    # Keep comment availability information
    record["comments_status"] = (
        "Disabled" if comments_disabled else "Available"
    )

    total_ratings = likes + dislikes

    # Check ratings availability
    if ratings_disabled or total_ratings == 0:
        return "Unrated"

    # Calculate like ratio
    like_ratio = likes / total_ratings

    # Classification
    if views >= 1000000 and like_ratio >= 0.90:
        return "Viral Hit"

    elif views >= 100000 and like_ratio < 0.60:
        return "Divisive"

    elif views >= 100000 and like_ratio >= 0.60:
        return "Steady Performer"

    else:
        return "Low Engagement"


print("Classification function created successfully!")

Classification function created successfully!


In [ ]:
sample = cleaned_records[0]

result = classify_video(sample)

print("Video Title:", sample["title"])
print("Views:", sample["views"])
print("Classification:", result)

Video Title: Wild Winter Prank
Views: 84344
Classification: Low Engagement


In [ ]:
label_counts = {
    "Viral Hit": 0,
    "Steady Performer": 0,
    "Divisive": 0,
    "Low Engagement": 0,
    "Unrated": 0
}

for record in cleaned_records:

    label = classify_video(record)

    label_counts[label] += 1


print("VIDEO CLASSIFICATION REPORT")

for label, count in label_counts.items():
    print(f"{label}: {count}")

print("Total classified:", sum(label_counts.values()))

VIDEO CLASSIFICATION REPORT
Viral Hit: 8864
Steady Performer: 27904
Divisive: 180
Low Engagement: 8738
Unrated: 544
Total classified: 46230


In [ ]:
sample = cleaned_records[0]

views = sample["views"]
likes = sample["likes"]
dislikes = sample["dislikes"]

total_ratings = likes + dislikes

print("Video:", sample["title"])
print("Views:", views)
print("Likes:", likes)
print("Dislikes:", dislikes)

if sample["ratings_disabled"] or total_ratings == 0:
    print("Branch: Unrated")

else:
    like_ratio = likes / total_ratings

    print("Like ratio:", round(like_ratio, 3))

    if views >= 1000000 and like_ratio >= 0.90:
        print("Branch: Viral Hit")

    elif views >= 100000 and like_ratio < 0.60:
        print("Branch: Divisive")

    elif views >= 100000 and like_ratio >= 0.60:
        print("Branch: Steady Performer")

    else:
        print("Branch: Low Engagement")

Video: Wild Winter Prank
Views: 84344
Likes: 2155
Dislikes: 163
Like ratio: 0.93
Branch: Low Engagement


In [ ]:
def count_by_field(records, field):

    counts = {}

    for record in records:

        value = record.get(field) or "Unknown"

        counts[value] = counts.get(value, 0) + 1

    return counts


# Count channel appearances
channel_summary = count_by_field(
    cleaned_records, "channel_title"
)

# Count country records
country_summary = count_by_field(
    cleaned_records, "country"
)

print("Total channels:", len(channel_summary))
print("Country summary:", country_summary)

Total channels: 438
Country summary: {'GB': 15075, 'IN': 15075, 'US': 16080}


Refactoring:

I converted repeated dictionary counting logic into a reusable function called count_by_field().

This reduces duplicate code and makes the program easier to maintain and reuse.